In [1]:
import sys

print(sys.executable)

c:\Users\sansk\anaconda3\envs\rag\python.exe


In [2]:
import torch
import sentence_transformers
import faiss
import pypdf

print("Environment ready!")

Environment ready!


# 01 - Embeddings and Semantic Retrieval

## ResearchRAG

The goal of this notebook is to understand the retrieval component
of a Retrieval-Augmented Generation (RAG) system.

We will:

1. Understand text embeddings
2. Generate embeddings using a Sentence Transformer
3. Measure semantic similarity
4. Build a FAISS vector index
5. Perform Top-K semantic retrieval

At the end of this notebook, we will have implemented the
retrieval foundation of RAG from scratch.

## 1. Text Embeddings

An embedding converts text into a numerical vector that captures
semantic information.

For example:

"Autonomous vehicles use computer vision."

is converted into something like:

[0.12, -0.43, 0.81, ...]

The individual numbers do not have a simple human interpretation.

Instead, the complete vector represents the semantic characteristics
of the sentence.

The important property for RAG is:

Semantically similar text → similar vectors

This allows us to perform semantic search.

In [6]:
pip install pandas

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------- ----------------------------- 2.6/9.9 MB 13.7 MB/s eta 0:00:01
   ------------------------ --------------- 6.0/9.9 MB 16.8 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 16.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import numpy as np
import pandas as pd

import torch
import faiss

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("FAISS:", faiss.__version__)

print("\nEnvironment is ready!")


Python: 3.11.16 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:36:16) [MSC v.1942 64 bit (AMD64)]
PyTorch: 2.14.0+cpu
FAISS: 1.15.1

Environment is ready!


In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully!


In [4]:
documents = [
    "Autonomous vehicles use computer vision to understand their surroundings.",
    "Self-driving cars use cameras and sensors to perceive the road.",
    "Deep learning models are widely used for image recognition.",
    "The Transformer architecture uses attention mechanisms.",
    "Retrieval augmented generation combines retrieval with language models.",
    "Vector databases allow efficient similarity search over embeddings.",
    "I enjoy playing basketball with my friends.",
    "The capital of France is Paris."
]

for i, document in enumerate(documents):
    print(f"{i}: {document}")

0: Autonomous vehicles use computer vision to understand their surroundings.
1: Self-driving cars use cameras and sensors to perceive the road.
2: Deep learning models are widely used for image recognition.
3: The Transformer architecture uses attention mechanisms.
4: Retrieval augmented generation combines retrieval with language models.
5: Vector databases allow efficient similarity search over embeddings.
6: I enjoy playing basketball with my friends.
7: The capital of France is Paris.


In [5]:
embeddings = model.encode(
    documents,
    convert_to_numpy=True
)

print("Embedding shape:", embeddings.shape)

Embedding shape: (8, 384)


In [6]:
print(embeddings[0])

[ 4.08642553e-02  1.44931395e-02  1.28460154e-02 -5.03627732e-02
  6.04935288e-02 -5.21980971e-02  8.71659890e-02 -2.39996538e-02
  3.41608562e-02 -5.82759269e-03 -1.19272498e-02 -3.12057454e-02
 -1.06789283e-02 -1.87371746e-02  1.94331892e-02 -2.95856576e-02
  2.27973405e-02  3.56961861e-02 -6.86976165e-02 -3.58137651e-03
 -1.99054796e-02 -4.25069667e-02 -2.31680535e-02  2.14358605e-02
 -4.70737815e-02  1.21852472e-01  2.74670459e-02  4.30021761e-03
 -2.90393122e-02 -5.64176738e-02 -4.92928457e-03 -7.08936993e-03
  7.19373152e-02  3.65184471e-02 -2.32636780e-02 -2.93671321e-02
 -4.22626063e-02  3.58752869e-02 -2.73592770e-02 -4.95027229e-02
 -3.69757190e-02 -2.62144469e-02  1.50751183e-02 -2.39112303e-02
  8.41575190e-02  9.21946838e-02  5.84611483e-03 -1.16164666e-02
  6.37159944e-02 -4.60421108e-02 -7.61941299e-02 -1.71703175e-02
 -7.65160937e-03 -7.91476145e-02 -8.45190510e-02  9.49385539e-02
 -1.44684967e-02 -5.50362840e-02  3.92384268e-02  2.78821979e-02
  8.17314014e-02 -3.07774

compare 2 sentences 


In [7]:
sentence_a = "Autonomous vehicles use computer vision."
sentence_b = "Self-driving cars use cameras to understand the road."

embedding_a = model.encode([sentence_a])
embedding_b = model.encode([sentence_b])

similarity = cosine_similarity(
    embedding_a,
    embedding_b
)

print("Cosine similarity:", similarity[0][0])

Cosine similarity: 0.6496493


In [8]:
sentence_c = "I enjoy playing basketball."

embedding_c = model.encode([sentence_c])

similarity_unrelated = cosine_similarity(
    embedding_a,
    embedding_c
)

print("Similarity with unrelated sentence:", similarity_unrelated[0][0])

Similarity with unrelated sentence: 0.08094613


similairty matrix 

In [9]:
similarity_matrix = cosine_similarity(embeddings)

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (8, 8)


In [10]:
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=[f"Doc {i}" for i in range(len(documents))],
    columns=[f"Doc {i}" for i in range(len(documents))]
)

similarity_df.round(2)

,Doc 0,Doc 1,Doc 2,Doc 3,Doc 4,Doc 5,Doc 6,Doc 7
Doc 0,1.00,0.69,0.46,0.26,0.02,0.11,0.07,0.04
Doc 1,0.69,1.00,0.29,0.23,-0.01,-0.03,-0.01,0.07
Doc 2,0.46,0.29,1.00,0.24,0.09,0.29,0.02,-0.00
Doc 3,0.26,0.23,0.24,1.00,0.14,0.03,0.01,0.04
Doc 4,0.02,-0.01,0.09,0.14,1.00,0.29,0.08,-0.01
Doc 5,0.11,-0.03,0.29,0.03,0.29,1.00,0.03,-0.02
Doc 6,0.07,-0.01,0.02,0.01,0.08,0.03,1.00,-0.06
Doc 7,0.04,0.07,-0.00,0.04,-0.01,-0.02,-0.06,1.00


## 2. FAISS Vector Search

FAISS (Facebook AI Similarity Search) is a library for efficient
similarity search over vectors.

Our embeddings currently exist in memory:

Documents → Embeddings

FAISS allows us to create an index over those vectors and efficiently
retrieve vectors that are most similar to a query vector.

In [11]:
embedding_dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dimension)

print("Embedding dimension:", embedding_dimension)
print("Vectors in index:", index.ntotal)

Embedding dimension: 384
Vectors in index: 0


Add embeddings to FAISS

In [12]:
embeddings_f32 = embeddings.astype("float32")

index.add(embeddings_f32)

print("Vectors in index:", index.ntotal)

Vectors in index: 8


perform first retrieval

In [13]:
query = "How do self-driving cars understand the road?"

In [14]:
query_embedding = model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

In [15]:
k = 3

scores, indices = index.search(
    query_embedding,
    k
)

print("Scores:")
print(scores)

print("\nIndices:")
print(indices)

Scores:
[[0.8101293  0.611158   0.18967438]]

Indices:
[[1 0 2]]


Show the retrieved documents

In [16]:
for rank, (score, idx) in enumerate(
    zip(scores[0], indices[0]),
    start=1
):
    print(f"Rank {rank}")
    print(f"Score: {score:.4f}")
    print(f"Document: {documents[idx]}")
    print("-" * 80)

Rank 1
Score: 0.8101
Document: Self-driving cars use cameras and sensors to perceive the road.
--------------------------------------------------------------------------------
Rank 2
Score: 0.6112
Document: Autonomous vehicles use computer vision to understand their surroundings.
--------------------------------------------------------------------------------
Rank 3
Score: 0.1897
Document: Deep learning models are widely used for image recognition.
--------------------------------------------------------------------------------


In [17]:
def retrieve(query, model, index, documents, k=3):
    """
    Retrieve the top-k most relevant documents for a query.
    """

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "document": documents[idx],
            "score": float(score)
        })

    return results

In [18]:
query = "What is retrieval augmented generation?"

results = retrieve(
    query,
    model,
    index,
    documents,
    k=3
)

for i, result in enumerate(results, start=1):
    print(f"Rank {i}")
    print(f"Score: {result['score']:.4f}")
    print(result["document"])
    print()

Rank 1
Score: 0.7435
Retrieval augmented generation combines retrieval with language models.

Rank 2
Score: 0.2553
The Transformer architecture uses attention mechanisms.

Rank 3
Score: 0.1828
Vector databases allow efficient similarity search over embeddings.



try different queries 

In [19]:
queries = [
    "How do autonomous vehicles see the environment?",
    "What is RAG?",
    "How does vector search work?",
    "What is the capital of France?",
    "What is a Transformer?"
]

for query in queries:
    print("=" * 80)
    print("QUERY:", query)
    
    results = retrieve(
        query,
        model,
        index,
        documents,
        k=2
    )

    for result in results:
        print(
            f"{result['score']:.4f} | "
            f"{result['document']}"
        )

QUERY: How do autonomous vehicles see the environment?
0.7244 | Autonomous vehicles use computer vision to understand their surroundings.
0.6282 | Self-driving cars use cameras and sensors to perceive the road.
QUERY: What is RAG?
0.0914 | Retrieval augmented generation combines retrieval with language models.
0.0288 | Vector databases allow efficient similarity search over embeddings.
QUERY: How does vector search work?
0.4886 | Vector databases allow efficient similarity search over embeddings.
0.2494 | Retrieval augmented generation combines retrieval with language models.
QUERY: What is the capital of France?
0.8790 | The capital of France is Paris.
0.0691 | Self-driving cars use cameras and sensors to perceive the road.
QUERY: What is a Transformer?
0.6142 | The Transformer architecture uses attention mechanisms.
0.1363 | Deep learning models are widely used for image recognition.


## Key Observations

We have implemented semantic retrieval without an LLM.

The pipeline is:

Query
↓
Query Embedding
↓
Vector Similarity Search
↓
Top-K Relevant Documents

Important concepts learned:

- Embeddings represent semantic information as vectors.
- Similar meanings tend to have similar embeddings.
- Cosine similarity measures similarity between vectors.
- FAISS provides efficient vector similarity search.
- Top-K retrieval returns the most similar documents.

However, this is NOT complete RAG yet.

We still need:

Documents
↓
Text Extraction
↓
Chunking
↓
Embeddings
↓
Vector Store
↓
Retrieval
↓
LLM
↓
Grounded Answer

In [20]:
print("Documents:", len(documents))
print("Embedding dimension:", embeddings.shape[1])
print("FAISS vectors:", index.ntotal)

Documents: 8
Embedding dimension: 384
FAISS vectors: 8
